# EDA: Data describing a predatory fish invasion on a riverine food web Data Set

### *John Tran — SEIS 631*

## Setup and loading

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
df = pd.read_csv("sia_dat.csv")
print(df.dtypes)

sample_id          str
site_number      int64
site_name          str
species            str
species_id         str
flathead           str
d_15N          float64
d_13C          float64
TN             float64
TC             float64
university         str
length_mm      float64
dtype: object


In [15]:
df.head()

,sample_id,site_number,site_name,species,species_id,flathead,d_15N,d_13C,TN,TC,university,length_mm
0,SIA1-001,1,Clemson Island,Algae,ALGE,Y,5.79,-21.20,1.05,9.50,PSU,NaN
1,SIA1-002,1,Clemson Island,Algae,ALGE,Y,6.19,-21.48,0.60,7.17,PSU,NaN
2,SIA1-003,1,Clemson Island,Algae,ALGE,Y,6.29,-20.68,0.71,8.00,PSU,NaN
3,SIA1-004,1,Clemson Island,Algae,ALGE,Y,6.28,-20.28,0.62,5.52,PSU,NaN
4,SIA1-005,1,Clemson Island,Algae,ALGE,Y,6.21,-18.52,1.34,11.57,PSU,NaN


In [ ]:
df.sort_values("date", ascending=True)

,sample_id,site_number,site_name,species,species_id,flathead,d_15N,d_13C,TN,TC,university,length_mm
512,SUS-241,1,Clemson Island,Flathead,FHCF,Y,14.65,-21.78,14.46,44.60,PSU,772.0
513,SUS-242,1,Clemson Island,Flathead,FHCF,Y,14.36,-21.81,14.30,44.33,PSU,661.0
400,SIS1-003,1,Clemson Island,Snail,BEIN,Y,9.39,-18.75,5.79,34.93,PSU,10.5
401,SIS1-004,1,Clemson Island,Snail,BEIN,Y,9.80,-18.22,6.78,32.74,PSU,10.5
398,SIS1-001,1,Clemson Island,Snail,BEIN,Y,9.66,-18.52,8.30,36.32,PSU,10.5
...,...,...,...,...,...,...,...,...,...,...,...,...
46,SIA13-005,13,West Falls,Eelgrass,EELG,Y,9.10,-23.10,3.50,38.20,MSU,NaN
45,SIA13-004,13,West Falls,Eelgrass,EELG,Y,9.10,-16.20,3.10,40.80,MSU,NaN
44,SIA13-003,13,West Falls,Algae,ALGE,Y,6.50,-27.40,0.80,5.10,MSU,NaN
43,SIA13-002,13,West Falls,Algae,ALGE,Y,7.60,-27.50,0.80,4.20,MSU,NaN


In [ ]:
# clean data here

## 1. Provenance & scope

- **Where, Who, and How:** The data set is was from the Steam Official Page. Deepan.N, the owner of the dataset extracted it via web scrapping.
- **Why:** The owner writes that the purpose for the dataset is that the information from a game could provide useful insight of said game. However, I think the information across multiple of those games (published between 1970-2024) could provide insights on patterns like historical game trends and market changes.
Additionally, they also tell us to check out a Dashboard Webapp that used this dataset, so one could also say the purpose is to provide accessible data for developers that are looking to analyze video games.


## 2. Structure & quality

In [17]:
# Quality Check -- df.isna().sum()
df.isna().sum()

sample_id        0
site_number      0
site_name        0
species          0
species_id       0
flathead         0
d_15N            0
d_13C            0
TN               0
TC               0
university       0
length_mm      134
dtype: int64

##### Missing Values
By using `df.isna().sum()` I was looking for missing values across the columns to determine data prep such as imputation or dropping.

I conclude that while most columns such as url, publisher, and price are completely dont have missing values, columns like `pegi` and `categories` have a significant amount of missing values (4283 and 595 missing values respectively) that we will need to account for.

Side note: The dataset never explicitly said what `pegi` is but I've discovered it stands for Pan-European Game Information which is the rating system for video games. The `pegi_url` are links to the images of the actual rating while `pegi` itself containsthe content descriptors associated with that rating.

In [18]:
# Quality Check -- duplicate checks
df.duplicated().sum()

np.int64(0)

In [20]:
# See list of name

##### Duplicate Checks
By using `df.duplicated().sum()`, I was looking for identical rows across the dataset to check for accidental redundancies
I conclude that there are 148 duplicate rows in the dataset that should be reviewed incase I need to remove redundant data.
So by using `df[df.duplicated(keep=False)].sort_values("name")`, I reviewed the list of duplicates and my initial thought is that before analysis, these are duplicates that should be dropped.

In [ ]:
# Quality Check -- value_counts(dropna=False) on suspicious columns

# Date had some unusual date information like 🔵🔵💤  ⭐TBA⭐🖤Soon🖤
print(df["date"].value_counts(dropna=False))

In [ ]:
# Both categories and pegi had a concerning amount of missing values
print(df["categories"].value_counts(dropna=False))
print("-----------------------------------------------------------------------------")
print(df["pegi"].value_counts(dropna=False))

##### `value_counts(dropna=False)` on suspicious columns
I was looking for abnormalities within suspicious columns like date, categories, and pegi.
I conclude that the dataset uses a `-` as a place holder for missing values and oddly formatted data (particularly in date) and must be handled before analysis either by setting those values with null or by other means.

## 3. Distributions

I believe the key variables would be `price`, `date`, `user_reviews`

In [ ]:
# Distribution plots and summaries for your key variables.

print(df["price"].head(10))
print(df["price"].dtype)

# So because the data was web scraped, the text surround the price was also added
# I've noticed that priced games have their price between "$" and "Add to Cart"
#   while free games have "Play game at the end" or have "Demo"
#   then theres the - in place of null
# Extract listed dollar prices from the scraped text

# Replace the original price string to only include the number
# ex: Buy Fall Guys$19.99Add to Cart -> 19.99
temp["price"] = temp.price.str.extract(r"\$(\d+\.\d{2})")

# Convert price dtype from string to numeric
temp["price"] = pd.to_numeric(temp["price"], errors="coerce")


# Setting free/demo games to 0
free_games = df.price.str.contains("free to play|demo", case=False, na=False)
temp.loc[free_games, "price"] = 0

paid_prices = temp.loc[temp["price"] > 0, "price"]

paid_prices[paid_prices > 0].plot.hist(
    bins=30, edgecolor="black"
)

plt.title("Distribution of Paid Game Prices")
plt.xlabel("Price ($)")
plt.ylabel("Number of Games")
plt.show()

# I noticed the x-axis is goes all the way to $1000, so I'm printing out to see what is
#   so expensive and scewing the graph as such
print(temp.nlargest(10, "price")[["name", "price"]])

### Price (first look)
After converting `price` from a string to a numeric and setting free games and demo games as 0. I created a histogram plot to see its shape:

It seems that the Plot for priced games above is insanely right skewed. I noticed the x-axis goes all the way out to $1000.
Some of the highest-priced games also appear to have missing names. I think the best course of action is to get the 99th percentile as a cutoff to represent majority of the paid games.

In [ ]:

price_limit = paid_prices.quantile(0.99)

paid_prices[paid_prices <= price_limit].plot.hist(
    bins=30, edgecolor="black"
)

plt.title("Distribution of Paid Game Prices (< 99th %tile)")
plt.xlabel("Price ($)")
plt.ylabel("Number of Games")
plt.show()

print(paid_prices.describe())
print("Median:", paid_prices.median())
print("99th percentile:", price_limit)
print("Prices above 99th percentile:", (paid_prices > price_limit).sum())


### Price (Continued)
The histogram is still right skewed with most games are on the lower end of the prices. I believe that for skewed graphs, compared to the mean, the median is a better representation of data. The median price is \$5.99, compared with a mean of \$11.52, which means the expensive games pull the mean upward. Most of prices range from \$2.99 to \$13.99. The maximum price is \$962.60, which is why there is a long right tail in the first histogram. By limiting the second histogram to the 99th percentile (\$79.20), I excluded the most expesive 1\% of games to view the distribution of paid game prices more clearly.

In [ ]:
temp["publish_year"] = temp.date.dt.year

temp["publish_year"].hist(bins=30)

plt.title("Distribution of Game Released Dates")
plt.xlabel("Year")
plt.ylabel("Number of Games")
plt.show()


# I noticed a long tail since it stretches to 1970, but the histogram is clear enough to use as is
print(temp.publish_year.sort_values(ascending=True).head())
print(temp.publish_year.describe())

### Date
The histogram is left skewed with most games emerging in the 2000s onward. Just like price, compared to the mean, the median is a better representation of data. The median price is 2018, compared with a mean of 2016, which means the older games pull the mean downward. Most of game released dates range from 2016 to 2019. Both my print statements confirmed that the oldest game was published in 1970 and the latest in 2024 as of when the data was webscrapped.

In [ ]:
print(temp["user_reviews"].head())

# Review are formated like "Very Positive(1,096)- 94% of the 1,096 user reviews in the last 30 days are positive."
#   and "Mostly Negative(5,782)- 28% of the 5,782 user reviews for this game are positive."
# so I plan to use the percentages as positive reviews


print("------------------------------------")
percent_counts = df["user_reviews"].str.count("%").value_counts()
are_positive_counts = df["user_reviews"].str.count("are positive").value_counts()
negative_counts = df["user_reviews"].str.count("negative").value_counts()
print("Num of. '%' in a review", percent_counts, "\n")
print("Reviews that include 'are positive'", are_positive_counts, "\n")
print("Reviews that include 'negative'", negative_counts, "\n")

print("------------------------------------")
print("List of review without '%' or 'are positive'")
print(df.loc[
    df["user_reviews"].str.count("%") == 0,
    "user_reviews"
].head(20).to_list())
print(df.loc[
    df["user_reviews"].str.count("are positive") == 0,
    "user_reviews"
].head(20).to_list())

print("------------------------------------")
temp["positive_reviews"] = pd.to_numeric(
    df["user_reviews"].str.extract(r"(\d+)%")[0],
    errors="coerce"
)

temp["positive_reviews"].hist(bins=30)

plt.title("Distribution of Positive Review Percentages")
plt.xlabel("Positive Reviews (%)")
plt.ylabel("Number of Games")
plt.show()


print(temp.positive_reviews.describe())


### User Reviews
Review are formated like:
- "`Very Positive(1,096)- 94% of the 1,096 user reviews in the last 30 days are positive.`"
- "`Mostly Negative(5,782)- 28% of the 5,782 user reviews for this game are positive.`"

so I used the percentages as positive reviews.

To double check that I was using the right percent. Steam tends to use the same format but there could be a chance a review reads along the lines of "6% of reviews are negative but 94% are positive". So I checked for how many instances of "%" a row has and how many included "are positive" and how many included "negative".

Both "\%" and "are positive" came out to be the same instance of 1 in each review while none of them said "negative". So all each review were about positive reviews and mention one percentage so `[0]` can be used to find each percentage. Additionally there were some reviews with 0 mentions of `%` those were either ones that mentions "No user reviews" or "-" which is a fill in for null.

The histogram is left skewed with most games at least half their reviews being positive. Just like price, compared to the mean, the median is a better representation of data. The median positive review percentage is 80%, compared with a mean of 75.63%, which means the poorer quality games pull the mean downward. Most of game's positive review percentage range from 65% to 90%. The highest positive review is a 100% meaning that all recorded reviews were positive while the lowest amount was 0% having no postive recorded reviews to its name.

## 4. Relationships

*(Question set 10–11.)* The two or three most important relationships, shown and interpreted — and what's surprising.

I believe two important relationships could be Publishing Year vs. Positive Reviews and Positive Reviews for Free vs. Paid Games.

In [ ]:
# Relationship plots (scatterplots, grouped comparisons, crosstabs).
plt.scatter(
    temp["publish_year"],
    temp["positive_reviews"],
    alpha=0.2
)

plt.title("Release Year vs. Positive Review Percentage")
plt.xlabel("Release Year")
plt.ylabel("Positive Reviews (%)")
plt.show()


temp["price_type"] = temp["price"].apply(
    lambda x: "Free" if x == 0 else ("Paid" if x > 0 else None)
)
temp.boxplot(column="positive_reviews", by="price_type")
plt.title("Positive Reviews: Free vs. Paid Games")
plt.suptitle("")
plt.xlabel("Price Type")
plt.ylabel("Positive Reviews (%)")
plt.show()

### Release Year vs. Positive Review Percentage

The scatter plot shows that newer games generally tend to have higher positive-review percentages. This is not really surprising because newer games may benefit from improvements in game design and technology. However, there is still variation in review percentages across release years, so newer games do not automatically receive better reviews.

### Free vs. Paid Games

The box plot shows that free and paid games have similar median positive review percentages, both around 80%. Their first and third quartiles are also similar, suggesting that the middle 50% of games in both groups have comparable review percentages. Both groups have long lower whiskers and several outliers on the bottom, meaning some games receive much lower positive review percentages than the typical game. Paid games have more visible outliers, although might be because there are more paid games in the dataset. I surprised that free games don't seem to have a lower typical positive reviews than paid games since I assumed games with higher cost tend to be high quality.

## 5. Readiness

*(Question set 12–13.)* If an analyst wanted to run an analysis tomorrow, what would they still have to do? What analyses is this dataset ready to support?

The dataset is only partially ready for analysis. I cleaned the variables needed for my analyses using temporary variables, so the original dataset remains unchanged. An analyst would need to repeat these cleaning steps or permanently apply them before running the same analyses. While the cleaned variables are mostly ready to support analyses involving reviews and dates, many other columns still require preprocessing, as noted by the dataset creator. They state "Most of the columns contains valuable information that could give a better insight of the game, but cleaning them is required for most of them". Therefore, additional data cleaning would be needed before doing analyses using the full dataset.

## 6. Your questions (14, 15, …)

The dataset-specific questions this dataset forces that the starter set doesn't ask — at least two, numbered, with your answers. These join your personal EDA guide and travel with you to Module 6 and the final project.

1. Do free games receive different review scores than paid games?
- Based on the box plot comparison, free and paid games are very similar median in  their positive review percentages. Both of them were around 80%. Even though paid games show more outliers, the percentage does not appear substantially different between free and paid.

2. Is there a relationship between a game's release year and its positive review percentage?
- If newer games tend to receive higher review scores, possible explanations might be improvements in game development technology. However, the dataset does not contain information on advertising, development budgets, or game quality metrics, so it cannot determine the cause of any observed relationship. If additional data were available, I would be interested in exploring whether marketing efforts like advertisements, trailers, and teasers, influence user reviews.

## Reflection

A few paragraphs (see the README): what was confusing and how you resolved it; where AI helped; one suggestion you verified, corrected, narrowed, or rejected and how you checked it; what you can now explain or do without AI that you couldn't before; how long it took; and any unresolved questions. If you did not use AI, explain how you found help and verified your work instead.

**What was confusing and how you resolved it?** I was confused at the `3. Distribution` section when it asked for shape, center, spread, outliers. I referred looked back at my notes and figured out that shape meant how skewed it was and how it affected the mean.

**Where AI helped?** AI helped me with the plots and graphs, before using it I kept getting errors when writing it. The AI also added things like labels, opacity on the scatter plot, and graph outlines for my histogram which made it way more clear how the graph looked.

**One suggestion you verified, corrected, narrowed, or rejected and how you checked it** The one I checked with AI was my notes in case I was wrong. I also asked it how accurate my interpretations were and made modifications from there.

**What you can now explain or do without AI that you couldn't before** I can understand the box plots a little better now so I think maybe that.

**How long it took** I took about 3 days to do, working on it when I could. I don't think I currently have any unresolved questions but if any arises after submission, I will be sure to ask by email or in class.